# Text Simplification Evaluation Notebook
This notebook performs full EDA, runs a T5-small model on test samples, and calculates BLEU and Flesch scores.

In [ ]:
!pip install -q pandas matplotlib seaborn transformers torch sacrebleu textstat nltk
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sacrebleu.metrics import BLEU
import textstat
import os

sns.set(style="whitegrid")

## 1. Data Loading
Please ensure `test_file.csv` is uploaded to the same folder as this notebook.

In [ ]:
FILE_PATH = 'test_file.csv'

if os.path.exists(FILE_PATH):
    df = pd.read_csv(FILE_PATH)
    print(f"Loaded {len(df)} samples from {FILE_PATH}")
    # Standardize column names if necessary
    # df = df.rename(columns={'original': 'input_text', 'simplified': 'target_text'})
else:
    print(f"Error: {FILE_PATH} not found. Please upload it.")
    # Creating dummy data for demonstration if file is missing
    df = pd.DataFrame({
        'input_text': ["The atmospheric pressure is high.", "The quick brown fox jumps over the lazy dog."] * 25,
        'target_text': ["It is very windy.", "A fast fox jumped over a dog."] * 25
    })

df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
df['input_len'] = df['input_text'].apply(lambda x: len(str(x).split()))
df['target_len'] = df['target_text'].apply(lambda x: len(str(x).split()))
df['input_flesch'] = df['input_text'].apply(lambda x: textstat.flesch_reading_ease(str(x)))
df['target_flesch'] = df['target_text'].apply(lambda x: textstat.flesch_reading_ease(str(x)))

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Input Sentence Length Distribution
sns.histplot(df['input_len'], bins=20, kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of Input sentence Lengths')
axes[0, 0].set_xlabel('Word Count')

# Plot 2: Target Sentence Length Distribution
sns.histplot(df['target_len'], bins=20, kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Distribution of target sentence Lengths')
axes[0, 1].set_xlabel('Word Count')

# Plot 3: length Comparison (Input vs Target)
sns.scatterplot(x='input_len', y='target_len', data=df, ax=axes[1, 0], alpha=0.6)
axes[1, 0].plot([0, df['input_len'].max()], [0, df['input_len'].max()], '--r')
axes[1, 0].set_title('Input Length vs Target Length')
axes[1, 0].set_xlabel('Input Word Count')
axes[1, 0].set_ylabel('Target Word Count')

# Plot 4: Flesch Reading Ease Comparison
flesch_df = pd.melt(df[['input_flesch', 'target_flesch']], var_name='Type', value_name='Score')
sns.boxplot(x='Type', y='Score', data=flesch_df, ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Reading Ease Comparison (Higher = Easier)')

plt.tight_layout()
plt.show()

## 3. Model Loading & Inference

In [ ]:
MODEL_NAME = "t5-small"
print("Loading model...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on {device}")

In [ ]:
def simplify_text(text):
    input_text = "summarize: " + str(text).strip()
    inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    outputs = model.generate(inputs, max_length=150, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Run on 50 samples
eval_df = df.head(50).copy()
print("Running simplification on 50 samples...")
eval_df['generated_text'] = eval_df['input_text'].apply(simplify_text)
eval_df.head()

## 4. Scoring & Results

In [ ]:
bleu_metric = BLEU()

references = [[str(r) for r in eval_df['target_text'].tolist()]]
hypotheses = eval_df['generated_text'].tolist()

bleu_score = bleu_metric.corpus_score(hypotheses, references)
print(f"BLEU Score: {bleu_score.score:.2f}")

eval_df['gen_flesch'] = eval_df['generated_text'].apply(lambda x: textstat.flesch_reading_ease(str(x)))
avg_flesch = eval_df['gen_flesch'].mean()
print(f"Average Flesch Score (Generated): {avg_flesch:.2f}")
print(f"Average Flesch Score (Original): {eval_df['input_flesch'].mean():.2f}")

In [ ]:
eval_df.to_csv('evaluation_results.csv', index=False)
print("Results saved to evaluation_results.csv")